# 05 - Xây dựng đặc trưng

Dựa trên các phát hiện ở notebook 04, xây bảng đặc trưng cuối cùng cho Random
Forest: cắt bỏ giai đoạn "doanh số = 0 giả" trước khi cửa hàng thực sự bán từng
nhóm hàng, thêm đặc trưng thời gian, và đặc trưng trễ/trung bình trượt (lag/rolling)
tính đúng theo thứ tự thời gian để **không rò rỉ dữ liệu tương lai**.

Điểm mấu chốt: `train` và `test` được **gộp vào 1 khung thời gian liên tục** trước
khi tính lag/rolling, vì đặc trưng của test (16 ngày dự báo, 2017-08-16 → 08-31)
phải được tính từ lịch sử thật của train, sau đó mới tách lại.

In [ ]:
import sys
from pathlib import Path

THU_MUC_HIEN_TAI = Path.cwd()
GOC = THU_MUC_HIEN_TAI.parent if THU_MUC_HIEN_TAI.name == "notebooks" else THU_MUC_HIEN_TAI
sys.path.append(str(GOC / "src"))

from cau_hinh import DATA_PROCESSED, OUT_TABLES, tao_thu_muc
from spark_session import tao_spark
from pyspark.sql import functions as F, Window

tao_thu_muc()
spark = tao_spark("05_DacTrung")

# transactions bi loai: chi co o train, dung truc tiep se ro ri du lieu (xem canh bao o notebook 03)
df_train = spark.read.parquet(str(DATA_PROCESSED / "train_sach")).drop("transactions")
df_test = spark.read.parquet(str(DATA_PROCESSED / "test_sach")).drop("transactions")

so_dong_train_goc = df_train.count()
so_dong_test_goc = df_test.count()
print("So dong goc: train =", so_dong_train_goc, "| test =", so_dong_test_goc)

## 1. Gộp train + test thành 1 khung thời gian liên tục

In [ ]:
df_train = df_train.withColumn("tap", F.lit("train"))
df_test = df_test.withColumn("sales", F.lit(None).cast("double")).withColumn("tap", F.lit("test"))

cot_chung = df_train.columns
assert set(cot_chung) == set(df_test.columns)
df_all = df_train.select(cot_chung).unionByName(df_test.select(cot_chung))

print("So dong sau gop:", df_all.count(), "(ky vong:", so_dong_train_goc + so_dong_test_goc, ")")
assert df_all.count() == so_dong_train_goc + so_dong_test_goc

## 2. Cắt "doanh số = 0 giả" trước ngày mở bán thực tế

Thực hiện ở mức **(cửa hàng, nhóm hàng)** — chặt hơn phát hiện mức cửa hàng ở
notebook 04, vì 1 cửa hàng có thể bắt đầu bán 1 nhóm hàng cụ thể muộn hơn ngày mở
cửa. Tổ hợp chưa từng có doanh số > 0 (thực sự không bán mặt hàng đó) được giữ
nguyên toàn bộ — không bị nhầm là dữ liệu lỗi.

In [ ]:
w_nhom = Window.partitionBy("store_nbr", "family")
df_all = df_all.withColumn(
    "ngay_dau_co_doanh_so",
    F.min(F.when(F.col("sales") > 0, F.col("date"))).over(w_nhom)
)
so_dong_truoc_cat = df_all.count()
df_all = df_all.filter(F.col("ngay_dau_co_doanh_so").isNull() | (F.col("date") >= F.col("ngay_dau_co_doanh_so")))
so_dong_sau_cat = df_all.count()
print(f"So dong truoc/sau cat leading-zero: {so_dong_truoc_cat} -> {so_dong_sau_cat} "
      f"(da bo {so_dong_truoc_cat - so_dong_sau_cat} dong, "
      f"{100*(so_dong_truoc_cat - so_dong_sau_cat)/so_dong_truoc_cat:.1f}%)")
df_all = df_all.drop("ngay_dau_co_doanh_so")

## 3. Đặc trưng thời gian

In [ ]:
df_all = (df_all
          .withColumn("nam", F.year("date"))
          .withColumn("thang", F.month("date"))
          .withColumn("ngay_trong_thang", F.dayofmonth("date"))
          .withColumn("thu", F.dayofweek("date"))
          .withColumn("tuan_trong_nam", F.weekofyear("date"))
          .withColumn("la_cuoi_tuan", F.col("thu").isin(1, 7)))

## 4. Đặc trưng trễ (lag) và trung bình trượt theo `(store_nbr, family)`

Cửa sổ luôn chỉ nhìn về **quá khứ** (`rowsBetween(-N, -1)`, `lag(N)`), tuyệt đối
không dùng dữ liệu tại/sau ngày hiện tại → không rò rỉ.

In [ ]:
w_thoi_gian = Window.partitionBy("store_nbr", "family").orderBy("date")
for so_ngay_lag in [7, 14, 28]:
    df_all = df_all.withColumn(f"lag_{so_ngay_lag}", F.lag("sales", so_ngay_lag).over(w_thoi_gian))

w_roll_7 = w_thoi_gian.rowsBetween(-7, -1)
w_roll_28 = w_thoi_gian.rowsBetween(-28, -1)
df_all = (df_all
          .withColumn("tb_truot_7", F.avg("sales").over(w_roll_7))
          .withColumn("tb_truot_28", F.avg("sales").over(w_roll_28))
          .withColumn("do_lech_chuan_truot_28", F.stddev("sales").over(w_roll_28)))

df_all = df_all.withColumn("log_doanh_so", F.log1p("sales"))
df_all.printSchema()

## 5. Tách lại train / test và kiểm tra

In [ ]:
df_train_dac_trung = df_all.filter(F.col("tap") == "train").drop("tap")
df_test_dac_trung = df_all.filter(F.col("tap") == "test").drop("tap")

so_dong_train_final = df_train_dac_trung.count()
so_dong_test_final = df_test_dac_trung.count()
print("Train sau dac trung:", so_dong_train_final, "(da bo doan leading-zero)")
print("Test sau dac trung:", so_dong_test_final, "(phai bang goc:", so_dong_test_goc, ")")
assert so_dong_test_final == so_dong_test_goc, "Khong duoc mat dong nao o test!"

## 6. Giới hạn thực tế của đặc trưng lag ngắn hạn trên test

Test có 16 ngày (2017-08-16 → 08-31). Với `lag_7`/`lag_14` (< 16 ngày), càng về
cuối giai đoạn test, cửa sổ lag càng "chạm" vào các ngày test khác chưa biết doanh
số thật → giá trị null. `lag_28`/`tb_truot_28` (≥ 16 ngày) luôn nhìn về train nên
**không bao giờ null trên test** — đây là lý do nên ưu tiên đặc trưng cửa sổ dài
khi huấn luyện mô hình dự báo nhiều bước ở notebook 06.

In [ ]:
cac_cot_lag = ["lag_7", "lag_14", "lag_28", "tb_truot_7", "tb_truot_28", "do_lech_chuan_truot_28"]
hang = []
for c in cac_cot_lag:
    so_null_train = df_train_dac_trung.filter(F.col(c).isNull()).count()
    so_null_test = df_test_dac_trung.filter(F.col(c).isNull()).count()
    hang.append((c, so_null_train, so_dong_train_final, so_null_test, so_dong_test_final))
    print(f"  null({c:<24}): train={so_null_train:>7}/{so_dong_train_final} "
          f"test={so_null_test:>6}/{so_dong_test_final}")

import pandas as pd
bang_null = pd.DataFrame(hang, columns=["dac_trung", "null_train", "tong_train", "null_test", "tong_test"])
bang_null.to_csv(OUT_TABLES / "dac_trung_ty_le_null.csv", index=False, encoding="utf-8-sig")

**Lưu ý cho notebook 06:** các giá trị null còn lại (lịch sử chưa đủ ở đầu chuỗi
sau khi cắt leading-zero, hoặc lag ngắn hạn ở cuối test) sẽ được xử lý trong pipeline
huấn luyện (impute), không xử lý ở đây — vì chiến lược impute gắn với lựa chọn mô hình.

## 7. Ghi kết quả

In [ ]:
df_train_dac_trung.write.mode("overwrite").parquet(str(DATA_PROCESSED / "train_dac_trung"))
df_test_dac_trung.write.mode("overwrite").parquet(str(DATA_PROCESSED / "test_dac_trung"))
print("Da ghi:", DATA_PROCESSED / "train_dac_trung")
print("Da ghi:", DATA_PROCESSED / "test_dac_trung")

kt = spark.read.parquet(str(DATA_PROCESSED / "test_dac_trung"))
print("Doc lai test_dac_trung, so dong:", kt.count())
assert kt.count() == so_dong_test_goc

In [ ]:
spark.stop()